# Beyond Bits — End-to-End Pipeline (Fully Integrated, Fixed)

Run every cell top-to-bottom, in ONE sitting, in order. Do not reuse files from a previous run.

**Folder layout expected:** this notebook, `RTL/*.v` files, and `LTSpice/Modulator_with_demod.asc` should all be reachable via the relative paths below. Adjust `RTL_DIR` if needed.

**Requires:** Icarus Verilog (`iverilog`, `vvp`) installed and on your PATH. On Ubuntu/WSL: `sudo apt install iverilog`. On Windows, install from https://bleyer.org/icarus/ and add it to PATH, or use ModelSim/Vivado manually for the RTL steps instead (see README).

In [ ]:
import subprocess, os

TEXT_INPUT = "hello world"
RTL_DIR = "../RTL"   # adjust if your folder layout differs

def run_verilog(sources, sim_name):
    """Compiles and runs a Verilog testbench via Icarus Verilog. Raises on failure."""
    compile_cmd = ["iverilog", "-o", sim_name] + sources
    r = subprocess.run(compile_cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"iverilog compile failed:\n{r.stderr}")
    r = subprocess.run(["vvp", sim_name], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"vvp simulation failed:\n{r.stderr}")


### Step 1: Text -> bits.txt

In [ ]:
def step1_generate_input():
    print("\n[1/9] Generating input bits...")
    binary = ''.join(format(ord(c), '08b') for c in TEXT_INPUT)
    with open("bits.txt", "w") as f:
        f.write('\n'.join(binary) + '\n')
    print(f"      Text: '{TEXT_INPUT}'  ->  {len(binary)} bits written to bits.txt")
    return binary

step1_generate_input()


### Step 2: RTL compression (bits.txt -> compressed.txt)
Runs `rle_compressor.v` + the race-fixed testbench automatically.

In [ ]:
print("[2/9] Running RTL compressor (Icarus Verilog)...")
run_verilog([f"{RTL_DIR}/rle_compressor.v", f"{RTL_DIR}/tb_compressor_fixed.v"], "compress_sim")


### Step 3: Pad + Hamming(7,4) encode (compressed.txt -> encoded_clean.txt)
ECC is required per SOI'26 Pipeline Enhancement #3. Padding to a multiple of 4 is recorded so it can be trimmed off later.

In [ ]:
def prepare_and_encode_ecc():
    print("\n[3/9] Padding + Hamming-encoding compressed.txt...")
    bits = [l.strip() for l in open("compressed.txt") if l.strip()]
    pad = (4 - (len(bits) % 4)) % 4
    padded = bits + ['0'] * pad
    with open("ecc_encode_input.txt", "w") as f:
        f.write('\n'.join(padded) + '\n')
    with open("ecc_pad.txt", "w") as f:
        f.write(str(pad))
    print(f"      {len(bits)} compressed bits -> padded to {len(padded)} (added {pad} pad bits)")
    return pad

prepare_and_encode_ecc()
run_verilog([f"{RTL_DIR}/hamming74_enc.v", f"{RTL_DIR}/tb_ecc_encode.v"], "ecc_encode_sim")


### Step 4: Generate signal.pwl for LTSpice

In [ ]:
V_HIGH = 5.0
BIT_PERIOD = 1e-6   # 1 microsecond per bit -- MUST match Step 6 sampling below

def generate_pwl(bits_path="encoded_clean.txt", out_path="signal.pwl"):
    print("\n[4/9] Generating PWL file for LTSpice...")
    bits = ''.join(l.strip() for l in open(bits_path) if l.strip())
    lines = []
    t = 0.0
    for bit in bits:
        v = V_HIGH if bit == '1' else 0.0
        lines.append(f"{t:.6e} {v:.1f}")
        t += BIT_PERIOD
    lines.append(f"{t:.6e} 0.0")
    with open(out_path, "w") as f:
        f.write('\n'.join(lines))
    print(f"      {len(bits)} bits -> {len(lines)} PWL points, {t*1e6:.1f} us total")
    print(f"      NOTE: your LTSpice .tran must simulate at least {t*1e6:.0f} us "
          f"(Modulator_with_demod.asc already uses 450us)")

generate_pwl()


### Step 5: STOP -- run LTSpice now

1. Open `LTSpice/Modulator_with_demod.asc`
2. Confirm `signal.pwl` is in the same folder (relative path already fixed)
3. Run the simulation
4. **File -> Export data as text**, export `V(DEMOD_OUT)` (not `V(ch_out)` -- that's the raw noisy carrier, DEMOD_OUT is the actual demodulated envelope)
5. Save as `demod_output.txt` in this notebook's folder

Continue to Step 6 once you have that file.

### Step 6: Sample the analog waveform back into digital bits

In [ ]:
import numpy as np

def parse_ltspice_wave_to_bits(spice_file="demod_output.txt", bit_period=BIT_PERIOD, threshold=0.4, out_path="demod_bits.txt"):
    print("\n[6/9] Reading analog waveform from LTSpice and thresholding...")
    time_axis, v_ch = np.loadtxt(spice_file, skiprows=1, unpack=True)
    total_duration = time_axis[-1]
    num_bits = int(round(total_duration / bit_period))
    bits = []
    for i in range(num_bits):
        sample_time = (i * bit_period) + (bit_period / 2.0)
        idx = np.abs(time_axis - sample_time).argmin()
        bits.append("1" if abs(v_ch[idx]) >= threshold else "0")
    with open(out_path, "w") as f:
        f.write('\n'.join(bits) + '\n')
    print(f"      Sampled {len(bits)} bits from the waveform -> {out_path}")
    return bits

parse_ltspice_wave_to_bits()


### Step 7: Hamming decode (detects + corrects any channel-noise bit flips)

In [ ]:
print("[7/9] Running RTL Hamming decoder...")
run_verilog([f"{RTL_DIR}/hamming74_dec.v", f"{RTL_DIR}/tb_ecc_decode.v"], "ecc_decode_sim")
print(open("ecc_decode_report.txt").read())


### Step 8: Trim padding, then RTL decompression

In [ ]:
def trim_and_prepare_decompress():
    print("\n[8/9] Trimming ECC padding before decompression...")
    pad = int(open("ecc_pad.txt").read())
    decoded = [l.strip() for l in open("decoded_bits.txt") if l.strip()]
    trimmed = decoded[:len(decoded) - pad] if pad else decoded
    with open("final_compressed_recovered.txt", "w") as f:
        f.write('\n'.join(trimmed) + '\n')
    print(f"      Trimmed {pad} pad bits -> {len(trimmed)} bits ready for decompression")

trim_and_prepare_decompress()
run_verilog([f"{RTL_DIR}/decompressor.v", f"{RTL_DIR}/tb_decompressor_final.v"], "final_decompress_sim")


### Step 9: Final verification

In [ ]:
print("[9/9] Converting recovered bits back to text...")
bits = ''.join(l.strip() for l in open("final_recovered_bits.txt") if l.strip())
chars = []
for i in range(0, len(bits), 8):
    byte = bits[i:i+8]
    if len(byte) == 8:
        val = int(byte, 2)
        if 32 <= val <= 126:
            chars.append(chr(val))
final_text = ''.join(chars)

print(f"\n======== END-TO-END VERIFICATION ========")
print(f"Expected Input:  '{TEXT_INPUT}'")
print(f"Recovered Text:  '{final_text}'")
print(f"Pipeline Integrity Verified: {final_text == TEXT_INPUT}")
print(f"=========================================")
